# Drowsiness Detection

Real-time drowsiness detection from a webcam. dlib finds the face and 68 facial landmarks, and the **Eye Aspect Ratio (EAR)** is computed from the eye landmarks. EAR drops when the eyes close, so if it stays below a threshold for enough consecutive frames, the notebook shows a "WAKE UP" warning and plays an alarm.

Before running, download the landmark model once with `python download_model.py`. Press `q` in the video window to quit.

# Imports

In [ ]:
import os
import cv2
import dlib
import imutils  # For resizing frames and converting dlib landmarks to numpy
from imutils import face_utils
import matplotlib.pyplot as plt
import numpy as np
from playsound import playsound
import threading

In [ ]:
def EAR_calc(eye): # We use EAR (Eye Aspect Ratio) to detect where the eyes are based on their location
    
    y1 = np.linalg.norm(eye[1] - eye[5])
    y2 = np.linalg.norm(eye[2] - eye[4])
    x1 = np.linalg.norm(eye[0] - eye[3])

    EAR = (y1+y2) / (2 * x1)

    return EAR

def play_sound(file):
    threading.Thread(target=playsound, args=(file,), daemon=True).start()

# Settings:
MODEL_PATH = 'Model/shape_predictor_68_face_landmarks.dat'  # get it with: python download_model.py
ALARM_SOUND = 'assets/alarm.wav'                            # swap in any .wav/.mp3 you like
blink_thresh = 0.25   # EAR below this means the eyes are (almost) closed
sleep_frames = 50     # consecutive closed-eye frames before the alarm triggers

# Variables:
count_drowsiness = 0
counter_EAR = 0
counter_EAR_list = []
avg_list = []
play_sound_bool = False

# Detectors and Landmarks
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"{MODEL_PATH} not found. Run `python download_model.py` first.")
detector = dlib.get_frontal_face_detector()
landmark_predict = dlib.shape_predictor(MODEL_PATH)

# Eye landmarks:
(L_start, L_end) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]
(R_start, R_end) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]
# WebCam
cam = cv2.VideoCapture(0)

In [ ]:
# By looking at the EAR, we see that when we blink or close our eyes,
# the graph falls down and it comes back up as we open our eyes.
# By calculating the frames that the graph is below our threshold 
# we evaluate if the user is sleeping or not by seeing if it surpasses the minimum frames for sleeping.

while True:

    ret, frame = cam.read()
    if not ret:  # camera not available or disconnected
        break

    video = imutils.resize(frame, width=420)
    gray = cv2.cvtColor(video, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)

    for face in faces:
        shape = landmark_predict(gray, face)
        shape = face_utils.shape_to_np(shape)

        lefteye = shape[L_start: L_end]
        righteye = shape[R_start: R_end]
        left_EAR = EAR_calc(lefteye)
        right_EAR = EAR_calc(righteye)
        avg = (left_EAR+right_EAR) / 2
        avg_list.append(avg)
        counter_EAR += 1
        counter_EAR_list.append(counter_EAR)
        
        if avg < blink_thresh:
            count_drowsiness += 1
            if count_drowsiness > sleep_frames:
                cv2.putText(frame, 'WAKE UP', (170, 250), cv2.FONT_HERSHEY_DUPLEX, 2, (0, 200, 0), 2)
                if play_sound_bool:
                    play_sound(ALARM_SOUND)
                play_sound_bool = False
        else:
            count_drowsiness = 0
            play_sound_bool = True

        
    cv2.imshow("Video", frame)
    if cv2.waitKey(1) == ord('q'):
        break
cam.release()
cv2.destroyAllWindows()

plt.plot(counter_EAR_list, avg_list)
plt.axhline(blink_thresh, color='r', linestyle='--', label='threshold')
plt.xlabel('Frame')
plt.ylabel('EAR')
plt.title('Eye Aspect Ratio over time')
plt.legend()
plt.show()